# RAG-система на основе обучающей выборки бизнес-новостей

Данный ноутбук реализует архитектуру Retrieval-Augmented Generation (RAG) в строгом соответствии с файлом **RAG.pdf**:
1. **Модель эмбеддингов**: BERT (bert-base-uncased), адаптированная для англоязычного бизнес-датасета.
2. **Чанкинг документов**: разбиение текстов на чанки длиной **500 символов** с перекрытием (**overlap**) **50 символов**.
3. **Векторная БД**: **FAISS** (faiss.IndexFlatL2) — хранилище векторов и индекс по евклидову расстоянию.
4. **ANN-поиск**: извлечение **top-5 - top-10** наиболее релевантных чанков по запросу.
5. **Сборка промпта и вызов LLM**: формирование контекста со строгой инструкцией против галлюцинаций + вызов LLM (GigaChat API) + возврат ответа с точным списком источников.


In [ ]:
# =====================================================================
# ШАГ 0: ИМПОРТЫ И НАСТРОЙКИ ОКРУЖЕНИЯ
# =====================================================================
import os
import re
import json
import uuid
import base64
import warnings
import urllib3
import requests
import numpy as np
import pandas as pd
import torch
from dotenv import load_dotenv
from transformers import AutoTokenizer, AutoModel
import faiss

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
warnings.filterwarnings("ignore")

# Загрузка .env переменных
load_dotenv(os.path.join(os.path.abspath(os.path.join(os.getcwd(), "..")), ".env"))
load_dotenv(".env")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🖥️ Устройство для BERT: {device}")


In [ ]:
# =====================================================================
# ШАГ 1: ЗАГРУЗКА ОБУЧАЮЩЕЙ ВЫБОРКИ (ГДЕ ТОЧНОСТЬ ОТЛИЧНАЯ)
# =====================================================================
data_paths = [
    "../data/processed/train_data.csv",
    "data/processed/train_data.csv",
    "../data/processed/business_data_cleaned.csv",
    "data/processed/business_data_cleaned.csv"
]
train_path = next((p for p in data_paths if os.path.exists(p)), None)
df_train = pd.read_csv(train_path)
print(f"✅ Загружено {len(df_train)} документов обучающей выборки.")
df_train.head(3)


In [ ]:
# =====================================================================
# ШАГ 2: ЧАНКИНГ (500 СИМВОЛОВ, OVERLAP 50 СИМВОЛОВ) ПО RAG.PDF
# =====================================================================
CHUNK_SIZE = 500
CHUNK_OVERLAP = 50
STEP = CHUNK_SIZE - CHUNK_OVERLAP

all_chunks = []
for doc_id, row in df_train.iterrows():
    text = str(row["text"]).strip()
    cat = row.get("category", "unknown")
    
    if len(text) <= CHUNK_SIZE:
        all_chunks.append({
            "chunk_id": len(all_chunks),
            "doc_id": doc_id,
            "category": cat,
            "text": text
        })
        continue
    
    for start in range(0, len(text), STEP):
        chunk_text = text[start:start + CHUNK_SIZE].strip()
        if len(chunk_text) >= 40:
            all_chunks.append({
                "chunk_id": len(all_chunks),
                "doc_id": doc_id,
                "category": cat,
                "text": chunk_text
            })

print(f"✅ Всего сформировано {len(all_chunks)} чанков (окно={CHUNK_SIZE}, шаг={STEP}).")
print("
Пример чанка 0:", all_chunks[0])


In [ ]:
# =====================================================================
# ШАГ 3: ЭМБЕДДИНГИ ЧЕРЕЗ BERT И СОЗДАНИЕ ВЕКТОРНОЙ БД В FAISS
# =====================================================================
MODEL_NAME = "bert-base-uncased"
print(f"🚀 Загрузка модели эмбеддингов: {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
bert = AutoModel.from_pretrained(MODEL_NAME).to(device)
bert.eval()

def embed_texts(texts, batch_size=64):
    embeddings = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        inputs = tokenizer(batch, padding=True, truncation=True, max_length=128, return_tensors="pt").to(device)
        with torch.no_grad():
            outputs = bert(**inputs)
            mask = inputs["attention_mask"].unsqueeze(-1).expand(outputs.last_hidden_state.size()).float()
            sum_embeddings = torch.sum(outputs.last_hidden_state * mask, dim=1)
            sum_mask = torch.clamp(mask.sum(dim=1), min=1e-9)
            mean_pooled = sum_embeddings / sum_mask
            normed = torch.nn.functional.normalize(mean_pooled, p=2, dim=1)
            embeddings.append(normed.cpu().numpy().astype("float32"))
    return np.vstack(embeddings)

# Проверяем кэш или рассчитываем эмбеддинги
cache_index_path = "../data/processed/faiss_rag_index.bin"
if not os.path.exists(cache_index_path):
    cache_index_path = "data/processed/faiss_rag_index.bin"

if os.path.exists(cache_index_path):
    print(f"⚡ Загрузка готового индекса FAISS из: {cache_index_path}")
    index = faiss.read_index(cache_index_path)
else:
    chunk_texts = [c["text"] for c in all_chunks]
    print(f"🧠 Генерация эмбеддингов для {len(chunk_texts)} чанков...")
    chunk_vectors = embed_texts(chunk_texts)
    index = faiss.IndexFlatL2(chunk_vectors.shape[1])
    index.add(chunk_vectors)
    faiss.write_index(index, cache_index_path)

print(f"✅ Индекс FAISS IndexFlatL2 готов! Всего векторов: {index.ntotal}")


In [ ]:
# =====================================================================
# ШАГ 4: ANN-ПОИСК В БАЗЕ (TOP 5-10 ЧАНКОВ ПО RAG.PDF)
# =====================================================================
def search_chunks(query, k=5):
    k = max(5, min(10, k))
    q_vec = embed_texts([query])
    distances, indices = index.search(q_vec, k)
    results = []
    for dist, idx in zip(distances[0], indices[0]):
        item = all_chunks[idx].copy()
        item["distance"] = float(dist)
        results.append(item)
    return results

test_q = "What happened with the interest rates decided by the central bank?"
found = search_chunks(test_q, k=5)
print(f"Найдено {len(found)} чанков для запроса: '{test_q}':
")
for i, r in enumerate(found, 1):
    print(f"[{i}] Doc #{r['doc_id']} | Cat: {r['category']} | L2: {r['distance']:.4f}")
    print(f"    {r['text'][:100]}...
")


In [ ]:
# =====================================================================
# ШАГ 5: СБОРКА ПРОМПТА И ВЫЗОВ GIGACHAT LLM (СО СТРОГОЙ ИНСТРУКЦИЕЙ)
# =====================================================================
def get_gigachat_token():
    client_id = os.getenv("GIGA_CLIENT_ID")
    client_secret = os.getenv("GIGA_CLIENT_SECRET")
    if not client_id or not client_secret:
        return None
    auth_b64 = base64.b64encode(f"{client_id}:{client_secret}".encode()).decode()
    headers = {
        "Authorization": f"Basic {auth_b64}",
        "RqUID": str(uuid.uuid4()),
        "Content-Type": "application/x-www-form-urlencoded"
    }
    resp = requests.post(
        "https://ngw.devices.sberbank.ru:9443/api/v2/oauth",
        headers=headers,
        data={"scope": os.getenv("GIGA_SCOPE", "GIGACHAT_API_PERS")},
        verify=False,
        timeout=10
    )
    return resp.json().get("access_token") if resp.status_code == 200 else None

def ask_rag(question, k=5):
    retrieved = search_chunks(question, k=k)
    context = "

".join([f"[{i+1}] {ch['text']}" for i, ch in enumerate(retrieved)])
    
    # Точный промпт из RAG.pdf
    prompt = (
        "Ответь на вопрос ТОЛЬКО по приведённым фрагментам. "
        "Если ответа в них нет — скажи 'в документах не найдено'.

"
        f"Фрагменты:
{context}

"
        f"Вопрос: {question}"
    )
    
    token = get_gigachat_token()
    headers = {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}
    data = {
        "model": "GigaChat",
        "messages": [
            {"role": "system", "content": "Ты строгий финансовый аналитик. Отвечай только на основе фрагментов."},
            {"role": "user", "content": prompt}
        ],
        "temperature": 0.1
    }
    resp = requests.post("https://gigachat.devices.sberbank.ru/api/v1/chat/completions", headers=headers, json=data, verify=False, timeout=20)
    
    answer = "Ошибка запроса"
    usage = {}
    if resp.status_code == 200:
        resp_json = resp.json()
        answer = resp_json["choices"][0]["message"]["content"]
        usage = resp_json.get("usage", {})
    
    print("=" * 75)
    print(f"❓ ВОПРОС: {question}")
    print("-" * 75)
    print(f"🤖 ОТВЕТ LLM:
{answer}")
    if usage:
        total = usage.get("total_tokens", "N/A")
        prompt_t = usage.get("prompt_tokens", "N/A")
        comp_t = usage.get("completion_tokens", "N/A")
        print("-" * 75)
        print(f"🪙 [ТОКЕНЫ] Использовано: {total} (Промпт: {prompt_t}, Ответ: {comp_t})")
    print("-" * 75)
    print(f"📚 ИСТОЧНИКИ (Top-{len(retrieved)} чанков):")
    for i, ch in enumerate(retrieved, 1):
        print(f"  [{i}] Doc #{ch['doc_id']} | {ch['category']} | L2={ch['distance']:.4f}")
    print("=" * 75)
    return answer, retrieved


In [ ]:
# Тестирование на реальном бизнес-вопросе (5 чанков)
ans1, src1 = ask_rag("What plans does Air India have for inducting aircraft and fleet expansion?", k=5)


In [ ]:
# Тестирование на внешнем вопросе (проверка защиты от галлюцинаций, должно вернуть "в документах не найдено")
ans2, src2 = ask_rag("Who won the UEFA Champions League football tournament in 2024?", k=5)
